## STEP 1: INITIAL COMPLIANCE & ENVIRONMENT CONFIG

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
# Core requirements
!pip install -U bitsandbytes>=0.46.1 torchao transformers datasets peft huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [4]:
import torch, gc
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments, 
    Trainer,
    pipeline
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, PeftConfig
from huggingface_hub import login, HfApi

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [ ]:
# Configuration constants matching your parameters
model_name = "Qwen/Qwen2.5-3B-Instruct"
local_adapter_dir = "my_qwen"
local_merged_dir = "my_qwen_merged"
hf_username = "usshaa"  # Replace with your actual Hugging Face username
repo_id = f"{hf_username}/my-qwen-rcm-validator"

In [ ]:
# Target query prompt structure to maintain throughout all test cycles
eval_messages = [
    {
        "role": "system", 
        "content": "You are a Revenue Cycle Management (RCM) compliance auditor. Analyze the clinical encounter notes, match the correct ICD-10 diagnosis codes, and highlight billing code variances in valid JSON format only."
    },
    {
        "role": "user", 
        "content": "Encounter Notes: Patient presents for evaluation of persistent hypertension. BP reading is 152/94 mmHg. Physician documents essential hypertension. Billing submitted code: I10."
    }
]

## STEP 2: TEST UNTRAINED BASE MODEL FOR TARGET QUERY

In [8]:
print("\n--- TEST 1: Querying Base Model Before Fine-Tuning ---")
base_tokenizer = AutoTokenizer.from_pretrained(model_name)
formatted_eval = base_tokenizer.apply_chat_template(eval_messages, tokenize=False, add_generation_prompt=True)


--- TEST 1: Querying Base Model Before Fine-Tuning ---


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [9]:
ask_base_llm = pipeline(
    task="text-generation",
    model=model_name,
    device="cuda"  # Match notebook cuda setting[cite: 3]
)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
# Base models usually return unstructured conversational strings rather than strict JSON
base_output = ask_base_llm(formatted_eval, max_new_tokens=150)[0]["generated_text"]
print("Base Model Output:\n", base_output)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Base Model Output:
 <|im_start|>system
You are a Revenue Cycle Management (RCM) compliance auditor. Analyze the clinical encounter notes, match the correct ICD-10 diagnosis codes, and highlight billing code variances in valid JSON format only.<|im_end|>
<|im_start|>user
Encounter Notes: Patient presents for evaluation of persistent hypertension. BP reading is 152/94 mmHg. Physician documents essential hypertension. Billing submitted code: I10.<|im_end|>
<|im_start|>assistant
```json
{
  "ICD-10_Codes": {
    "Correct_ICD_10_Codes": [
      {
        "Diagnosis_Code": "I10",
        "ICD_10_Description": "Essential hypertension"
      }
    ],
    "Billing_Code_Variations": [
      {
        "Billing_Code": "I10",
        "Issue": "No variance found, but it's recommended to ensure that the patient's condition matches the documentation."
      }
    ]
  }
}
```


In [11]:
# Clear caches to manage VRAM aggressively
del ask_base_llm
del base_tokenizer
gc.collect()
torch.cuda.empty_cache()

## STEP 3: DATA PREPARATION & CHAT TOKENIZATION

In [16]:
import re

In [ ]:
print("\n--- STEP 3: Loading and Processing Task Dataset ---")
data_file_path = "/kaggle/input/datasets/ussha47/rcm-data/rcm_compliance_dataset.json"

try:
    # Attempt absolute standard load first
    with open(data_file_path, "r", encoding="utf-8") as f:
        json_list = json.load(f)
except json.JSONDecodeError:
    print("Standard JSON array parsing failed. Initiating line-by-line regex data recovery...")
    json_list = []
    
    # Line-by-line loader ignores global missing commas and handles raw line chunks safely
    with open(data_file_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
        
        # Split records based on boundaries of the chat layout objects
        records = re.split(r'\}\s*,\s*\{\s*"messages"', content)
        
        for i, record in enumerate(records):
            # Clean and rebuild standard object encapsulation shapes
            if not record.startswith('{'):
                record = '{"messages"' + record
            if not record.endswith('}'):
                if record.rstrip().endswith('},'):
                    record = record.rstrip()[:-1]
                else:
                    record = record + '}'
            
            # Remove outermost wrapper array tokens if present in splits
            if record.startswith('[') or record.startswith(' \n['):
                record = record.strip()[1:]
            if record.endswith(']'):
                record = record.strip()[:-1]
                
            try:
                parsed_obj = json.loads(record)
                json_list.append(parsed_obj)
            except Exception:
                continue  # Silently skip malformed entries to ensure training keeps moving

# Convert the safely recovered list into a Hugging Face Dataset format
train_dataset = Dataset.from_list(json_list)
raw_data = DatasetDict({"train": train_dataset})
print(f"Dataset successfully loaded! Active validated training records: {len(raw_data['train'])}")


--- STEP 3: Loading and Processing Task Dataset ---
Standard JSON array parsing failed. Initiating line-by-line regex data recovery...
Dataset successfully loaded! Active validated training records: 96


In [18]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess(sample):
    # Enforces standard Instruct style tag structures dynamically
    formatted_chat = tokenizer.apply_chat_template(sample["messages"], tokenize=False)
    
    tokenized = tokenizer(
        formatted_chat,
        max_length=256,        
        truncation=True,       
        padding="max_length",  
    )
    tokenized["labels"] = tokenized["input_ids"].copy()  # Duplicate token mappings into target labels
    return tokenized

In [20]:
data = raw_data.map(preprocess)  # Convert structural arrays into input tokens

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

## STEP 4: INTRODUCE 4-BIT QUANTIZATION & LORA

In [21]:
print("\n--- STEP 4: Initializing LoRA Optimization Profile ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)


--- STEP 4: Initializing LoRA Optimization Profile ---


In [22]:
# Load base model wrapped in the notebook's memory optimizations
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj"]  # Intercept target linear projection layers
)

In [24]:
model = get_peft_model(model, lora_config)  # Extract localized parameter weights

## STEP 5: MODEL TRAINING LOOP Execution

In [25]:
print("\n--- STEP 5: Executing Fine-Tuning Sequence ---")
training_args = TrainingArguments(
    output_dir="./trainer_output",
    num_train_epochs=3,            # Train over 3 baseline epochs
    learning_rate=0.001,           # Base optimization learning rates
    logging_steps=25,
    gradient_checkpointing=True,   # Minimize memory footings
    per_device_train_batch_size=1, # Strict low batch allocation to prevent T4 OOM issues
)


--- STEP 5: Executing Fine-Tuning Sequence ---


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data["train"]     # Map clean training partitions
)

In [27]:
trainer.train()  # Launch fine-tuning execution

Step,Training Loss
25,2.349451
50,0.934249
75,0.692158
100,0.628450
125,0.547800
150,0.539741
175,0.521846
200,0.550324
225,0.438963
250,0.386197


TrainOutput(global_step=288, training_loss=0.7164808478620317, metrics={'train_runtime': 257.811, 'train_samples_per_second': 1.117, 'train_steps_per_second': 1.117, 'total_flos': 1228580025532416.0, 'train_loss': 0.7164808478620317, 'epoch': 3.0})

In [28]:
# Save optimized fine-tuned components locally
trainer.save_model(local_adapter_dir)
tokenizer.save_pretrained(local_adapter_dir)

('my_qwen/tokenizer_config.json',
 'my_qwen/chat_template.jinja',
 'my_qwen/tokenizer.json')

## STEP 6: TEST LOCAL TRAINED ADAPTER WITH TARGET QUERY

In [ ]:
print("\n--- TEST 2: Evaluating Local Fine-Tuned Adapter ---")
# Reload clean objects to verify the generated adapter folder files
peft_config = PeftConfig.from_pretrained(local_adapter_dir)
base_validation = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path, trust_remote_code=True)
validation_model = PeftModel.from_pretrained(base_validation, local_adapter_dir)
validation_tokenizer = AutoTokenizer.from_pretrained(local_adapter_dir, trust_remote_code=True)


--- TEST 2: Evaluating Local Fine-Tuned Adapter ---


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [30]:
# Process verification inputs
inputs = validation_tokenizer(formatted_eval, return_tensors="pt").to(validation_model.device)
output = validation_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=150  # Keep output length consistent for structural JSON generation
)

In [31]:
print("Local Adapter Model Output:\n", validation_tokenizer.decode(output[0], skip_special_tokens=True))

# Clean GPU memory layers comprehensively before undertaking CPU operations
del validation_model
del base_validation
gc.collect()
torch.cuda.empty_cache()  # Clear memory spaces completely

Local Adapter Model Output:
 system
You are a Revenue Cycle Management (RCM) compliance auditor. Analyze the clinical encounter notes, match the correct ICD-10 diagnosis codes, and highlight billing code variances in valid JSON format only.
user
Encounter Notes: Patient presents for evaluation of persistent hypertension. BP reading is 152/94 mmHg. Physician documents essential hypertension. Billing submitted code: I10.
assistant
{
  "status": "APPROVED",
  "variance_found": false,
  "submitted_codes": [
    "I10"
  ],
  "correct_codes": [
    "I10"
  ],
  "justification": "The submitted code I10 (Essential hypertension) accurately reflects the physician's documented finding of essential hypertension based on the BP reading provided."
}


## STEP 7: MERGE WEIGHTS AND SAVE TO DISK

In [ ]:
print("\n--- STEP 7: Commencing Weight Unload and Merger ---")
# Re-load onto CPU explicitly to completely bypass GPU capacity overflows[cite: 3]
base_model_cpu = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="cpu", 
    trust_remote_code=True
)

In [ ]:
merged_model = PeftModel.from_pretrained(base_model_cpu, local_adapter_dir)
merged_model = merged_model.merge_and_unload()  # Lock down active LoRA parameters back into base layout

In [ ]:
# Export fully combined model binaries locally to storage partitions
merged_model.save_pretrained(local_merged_dir)
tokenizer.save_pretrained(local_merged_dir)

## STEP 8: EXPORT ARTIFACTS TO HUGGING FACE HUB

In [ ]:
print("\n--- STEP 8: Uploading Merged Weights to Hugging Face Hub ---")
# Relies on the authentication credentials imported via Kaggle secrets or direct login[cite: 3]
login() 

In [ ]:
api = HfApi()
# Initialize remote public cloud repository path[cite: 3]
api.create_repo(repo_id=repo_id, repo_type="model", private=False)

In [ ]:
# Upload the fully compiled target folder containing safe tensors directly[cite: 3]
api.upload_folder(
    folder_path=local_merged_dir,
    repo_id=repo_id,
    repo_type="model"
)

In [ ]:
# Clean storage metrics again to prepare for cloud inference deployment
del merged_model
del base_model_cpu
gc.collect()

## STEP 9: PIPELINE DOWNLOAD & FINAL REMOTE VERIFICATION

In [ ]:
print(f"\n--- TEST 3: Calling Merged Cloud Model From Remote Repository ({repo_id}) ---")
# Direct instantiation pulling from your fresh Hugging Face cloud folder path
cloud_tokenizer = AutoTokenizer.from_pretrained(repo_id)
cloud_model = AutoModelForCausalLM.from_pretrained(repo_id, device_map="auto")

In [ ]:
cloud_inputs = cloud_tokenizer(formatted_eval, return_tensors="pt").to(cloud_model.device)
cloud_output = cloud_model.generate(**cloud_inputs, max_new_tokens=150)

print("Final Cloud Merged Model Output:\n", cloud_tokenizer.decode(cloud_output[0], skip_special_tokens=True))
print("\nFine-tuning loop complete! The model now deterministically outputs strict compliance JSON arrays.")